In [1]:
!pip install -r requirements.txt

In [2]:
!pip install -U google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompa

In [5]:
from google import genai
from google.genai import types
import time
client = genai.Client(api_key='')

In [6]:
def call_gemini_1(
    prompt: str,
    model: str = "gemini-2.5-flash",
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_tokens: int = 256,
) -> tuple[str, float, dict]:
    """
    Gọi Google Gemini API, đo thời gian phản hồi và thống kê lượng token sử dụng.
    """
    # 1. Cấu hình các tham số sinh văn bản qua types.GenerateContentConfig
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_tokens,
    )

    # 2. Bắt đầu bấm giờ đo độ trễ (latency)
    start_time = time.perf_counter()

    # 3. Gửi request đến Gemini API
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=config,
    )

    # 4. Tính thời gian xử lý (giây)
    latency_seconds = time.perf_counter() - start_time

    # 5. Lấy nội dung trả về
    response_text = response.text or ""

    # 6. Trích xuất thông tin token từ usage_metadata
    usage_meta = response.usage_metadata
    usage = {
        "input_tokens": getattr(usage_meta, "prompt_token_count", 0) if usage_meta else 0,
        "output_tokens": getattr(usage_meta, "candidates_token_count", 0) if usage_meta else 0,
        "total_tokens": getattr(usage_meta, "total_token_count", 0) if usage_meta else 0,
    }

    return response_text, latency_seconds, usage

In [7]:
result = call_gemini_1("Thủ đô của Việt Nam là gì?")
print(result)

('Thủ đô của Việt Nam là **Hà Nội**.', 0.6021876690000454, {'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 46})


In [8]:
def call_gemini_2(
    prompt: str,
    model: str = "gemini-3.5-flash",
    temperature: float = 0.7,
    top_p: float = 0.9,
    max_tokens: int = 256,
) -> tuple[str, float, dict]:
    """
    Gọi Google Gemini API, đo thời gian phản hồi và thống kê lượng token sử dụng.
    """
    # 1. Cấu hình các tham số sinh văn bản qua types.GenerateContentConfig
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=top_p,
        max_output_tokens=max_tokens,
    )

    # 2. Bắt đầu bấm giờ đo độ trễ (latency)
    start_time = time.perf_counter()

    # 3. Gửi request đến Gemini API
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config=config,
    )

    # 4. Tính thời gian xử lý (giây)
    latency_seconds = time.perf_counter() - start_time

    # 5. Lấy nội dung trả về
    response_text = response.text or ""

    # 6. Trích xuất thông tin token từ usage_metadata
    usage_meta = response.usage_metadata
    usage = {
        "input_tokens": getattr(usage_meta, "prompt_token_count", 0) if usage_meta else 0,
        "output_tokens": getattr(usage_meta, "candidates_token_count", 0) if usage_meta else 0,
        "total_tokens": getattr(usage_meta, "total_token_count", 0) if usage_meta else 0,
    }

    return response_text, latency_seconds, usage

In [9]:
# Gọi trực tiếp để kiểm tra Task 2:
result = call_gemini_2("Thủ đô của Việt Nam là gì?")
print(result)

('Thủ đô của Việt Nam là **Hà Nội**.', 2.0641567640000176, {'input_tokens': 10, 'output_tokens': 12, 'total_tokens': 130})


In [10]:
# Bảng giá tham chiếu USD trên 1 triệu token
PRICING = {
    "gemini-2.5-flash": {
        "input_per_1m": 0.30,
        "output_per_1m": 2.50,
    },
    "gemini-3.5-flash": {
        "input_per_1m": 1.50,
        "output_per_1m": 9.00,
    },
}

def calculate_token_cost(input_tokens: int, output_tokens: int, model_name: str) -> float:
    """Tính toán chi phí USD dựa trên số lượng token và đơn giá trên 1M token."""
    rates = PRICING.get(model_name, {"input_per_1m": 0.0, "output_per_1m": 0.0})
    return (input_tokens * rates["input_per_1m"] + output_tokens * rates["output_per_1m"]) / 1_000_000

def compare_models(prompt: str) -> dict:
    """
    So sánh Gemini 2.5 Flash và Gemini 3.5 Flash với cùng một prompt,
    trả về dictionary gồm response, latency, cost và token usage.
    """
    # 1. Gọi mô hình Gemini 2.5 Flash
    g25_resp, g25_latency, g25_usage = call_gemini_1(prompt, model="gemini-2.5-flash")
    g25_in = g25_usage.get("input_tokens", 0)
    g25_out = g25_usage.get("output_tokens", 0)
    g25_cost = calculate_token_cost(g25_in, g25_out, "gemini-2.5-flash")

    # 2. Gọi mô hình Gemini 3.5 Flash
    g35_resp, g35_latency, g35_usage = call_gemini_2(prompt, model="gemini-3.5-flash")
    g35_in = g35_usage.get("input_tokens", 0)
    g35_out = g35_usage.get("output_tokens", 0)
    g35_cost = calculate_token_cost(g35_in, g35_out, "gemini-3.5-flash")

    # 3. Trả về cấu trúc dictionary so sánh
    return {
        "gemini_2_5_flash": {
            "response": g25_resp,
            "latency": g25_latency,
            "cost": g25_cost,
            "input_tokens": g25_in,
            "output_tokens": g25_out,
        },
        "gemini_3_5_flash": {
            "response": g35_resp,
            "latency": g35_latency,
            "cost": g35_cost,
            "input_tokens": g35_in,
            "output_tokens": g35_out,
        },
    }

In [11]:
# 1. Gọi hàm compare_models với một câu prompt thử nghiệm
test_prompt = "Tại sao bầu trời lại có màu xanh? Trả lời ngắn trong 2 câu."
results = compare_models(test_prompt)

# 2. In bảng tổng hợp các chỉ số (Latency, Cost, Tokens)
print("=" * 75)
print(f"{'MÔ HÌNH':<18} | {'ĐỘ TRỄ (s)':<12} | {'CHI PHÍ ($)':<12} | {'INPUT TOK':<11} | {'OUTPUT TOK':<11}")
print("=" * 75)

for model_key, data in results.items():
    print(
        f"{model_key:<18} | "
        f"{data['latency']:<12.3f} | "
        f"${data['cost']:<11.6f} | "
        f"{data['input_tokens']:<11} | "
        f"{data['output_tokens']:<11}"
    )

print("=" * 75)

# 3. In chi tiết phản hồi (response) từ từng mô hình
print("\n" + "=" * 30 + " NỘI DUNG PHẢN HỒI " + "=" * 30)
for model_key, data in results.items():
    print(f"\n[MODEL: {model_key}]")
    print(data["response"].strip())
    print("-" * 75)

MÔ HÌNH            | ĐỘ TRỄ (s)   | CHI PHÍ ($)  | INPUT TOK   | OUTPUT TOK 
gemini_2_5_flash   | 1.564        | $0.000078    | 19          | 29         
gemini_3_5_flash   | 7.172        | $0.000092    | 19          | 7          

============================== NỘI DUNG PHẢN HỒI ==============================

[MODEL: gemini_2_5_flash]
Bầu trời có màu xanh vì ánh sáng mặt trời bị tán xạ bởi các phân tử khí và hạt nhỏ trong khí quyển Trái Đất
---------------------------------------------------------------------------

[MODEL: gemini_3_5_flash]
Bầu trời có màu xanh là
---------------------------------------------------------------------------


In [17]:
def streaming_chatbot() -> None:
    """
    Chatbot tương tác terminal: stream token, lưu trữ 3 lượt hội thoại gần nhất.
    """
    api_key = ""
    if not api_key:
        print("\033[93m[System Warning] GEMINI_API_KEY environment variable not set. Running in dummy mode.\033[0m")
        api_key = "mock-key"

    print("\n\033[94m====================================================")
    print("🤖 Vin Smart Future — Intelligent Chat Assistant")
    print("Powered by Google Gemini 2.5 Flash")
    print("Type 'quit' or 'exit' to end the session.")
    print("====================================================\033[0m\n")

    history = []

    while True:
        try:
            user_input = input("Nhập câu hỏi ").strip()
            if not user_input:
                continue

            if user_input.lower() in ["quit", "exit"]:
                print("\033[93mGoodbye!\033[0m")
                break

            # Giữ tối đa 3 turns (6 tin nhắn) gần nhất và gắn prompt mới vào cuối
            messages_to_send = []
            for h in history[-6:]:
                messages_to_send.append(h)
            messages_to_send.append({"role": "user", "text": user_input})

            print("\033[92mGemini:\033[0m ", end="", flush=True)
            full_reply = ""

            # Định dạng danh sách tin nhắn theo cấu trúc types.Content
            formatted_contents = []
            for msg in messages_to_send:
                formatted_contents.append(
                    types.Content(
                        role=msg["role"],
                        parts=[types.Part.from_text(text=msg["text"])]
                    )
                )

            # Khởi tạo client & stream phản hồi trực tiếp ra terminal
            chat_client = genai.Client(api_key=api_key)
            response_stream = chat_client.models.generate_content_stream(
                model="gemini-2.5-flash",
                contents=formatted_contents,
            )

            for chunk in response_stream:
                if chunk.text:
                    print(chunk.text, end="", flush=True)
                    full_reply += chunk.text

            print()

            # Lưu lại tương tác mới vào history
            history.append({"role": "user", "text": user_input})
            history.append({"role": "model", "text": full_reply})

        except KeyboardInterrupt:
            print("\n\033[93mGoodbye!\033[0m")
            break
        except Exception as e:
            print(f"\n\033[91m[Error]: {e}\033[0m")

# Kích hoạt vòng lặp chatbot chạy trực tiếp
streaming_chatbot()


🤖 Vin Smart Future — Intelligent Chat Assistant
Powered by Google Gemini 2.5 Flash
Type 'quit' or 'exit' to end the session.

Nhập câu hỏi thủ đô của Việt Nam
Gemini: Thủ đô của Việt Nam là **Hà Nội**.
Nhập câu hỏi quit
Goodbye!
